In [1]:
import pandas as pd
import numpy as np
import joblib
import random
from tqdm import tqdm
import plotly.express as px
from scipy.sparse import load_npz
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# movies = pd.read_csv(r"C:/Users/ACER/Desktop/Dataset/DATA/Movie/Moive lens/ml-latest/ml-latest/movies.csv")
#tags = pd.read_csv(r"C:\Users\ACER\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Python 3.12\filtered_tags_50_to_150.csv")
#ratings = pd.read_csv(r"C:\Users\ACER\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Python 3.12\filtered_ratings_50_to_150.csv")
#movies_tagged = pd.read_csv("saved_model/movies.csv") """

In [3]:
movies = pd.read_csv(r"C:\Users\ACER\Desktop\Dataset\DATA\Movie\Moive lens\ml-latest-small\ml-latest-small\movies.csv")
tags = pd.read_csv(r"C:\Users\ACER\Desktop\Dataset\DATA\Movie\Moive lens\ml-latest-small\ml-latest-small\tags.csv")
ratings = pd.read_csv(r"C:\Users\ACER\Desktop\Dataset\DATA\Movie\Moive lens\ml-latest-small\ml-latest-small\ratings.csv")
model_cf = joblib.load("svdpp_best_model_small.pkl")

tfidf_matrix = load_npz("models/tfidf_matrix.npz")
cosine_sim = np.load("models/cosine_sim.npy")
tfidf = joblib.load("models/tfidf_vectorizer.joblib")
movies_tagged = pd.read_csv("models/movies_with_content.csv")
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [4]:
# CF- Collaborative Filtering

In [5]:
def CF_recommendations(model, ratings_df, user_id, movies_df=None, n=10):
    
    if user_id not in ratings_df['userId'].unique():
        print(f"User ID {user_id} không tồn tại trong dữ liệu!")
        return pd.DataFrame()  # Trả về DataFrame rỗng
    
    # Phần còn lại giữ nguyên
    seen_movie_ids = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])
    all_movie_ids = set(ratings_df['movieId'].unique())
    unseen_movie_ids = all_movie_ids - seen_movie_ids

    predictions = [model.predict(user_id, movie_id) for movie_id in unseen_movie_ids]
    top_predictions = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]

    top_movies = []
    for pred in top_predictions:
        title = str(pred.iid)
        if movies_df is not None:
            title_result = movies_df[movies_df['movieId'] == int(pred.iid)]
            if not title_result.empty:
                title = title_result['title'].values[0]
        top_movies.append({
            'movieId': pred.iid,
            'title': title,
            'predicted_rating': pred.est
        })

    top_recs_df = pd.DataFrame(top_movies)

    print(f"\nTop {n} phim gợi ý cho user {user_id}:")
    for _, row in top_recs_df.iterrows():
        print(f"{row['title']} (ID: {row['movieId']}) - Dự đoán: {row['predicted_rating']:.2f}")

    return top_recs_df


In [6]:
user_id = random.choice(ratings['userId'].unique())

top_n = 10
top_recs_df = CF_recommendations(
    model=model_cf,
    ratings_df=ratings,
    user_id=user_id,
    movies_df=movies,
    n=top_n
)
import plotly.express as px
fig = px.bar(
    top_recs_df.sort_values("predicted_rating"),
    x="predicted_rating",
    y="title",
    orientation="h",
    title=f"Top {top_n} phim được gợi ý cho User {user_id}",
    labels={
        "predicted_rating": "Điểm dự đoán",
        "title": "Tên phim"
    },
    color="predicted_rating",
    color_continuous_scale="viridis"
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_range=[0, 5],
    template="plotly_white"
)
fig.show()


Top 10 phim gợi ý cho user 588:
Matrix, The (1999) (ID: 2571) - Dự đoán: 4.34
Fight Club (1999) (ID: 2959) - Dự đoán: 4.26
Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001) (ID: 4973) - Dự đoán: 4.17
Memento (2000) (ID: 4226) - Dự đoán: 4.16
Léon: The Professional (a.k.a. The Professional) (Léon) (1994) (ID: 293) - Dự đoán: 4.13
Godfather, The (1972) (ID: 858) - Dự đoán: 4.13
Reservoir Dogs (1992) (ID: 1089) - Dự đoán: 4.13
Saving Private Ryan (1998) (ID: 2028) - Dự đoán: 4.12
Monty Python and the Holy Grail (1975) (ID: 1136) - Dự đoán: 4.12
Hustler, The (1961) (ID: 3468) - Dự đoán: 4.11


In [7]:
# CBF - Content-Based Filtering

In [ ]:
def recommend_movies_for_user(user_id, ratings, movies, indices, cosine_sim, top_n=10):
    liked_movies = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4)]
    
    if liked_movies.empty:
        print(f"User {user_id} không có đánh giá >= 4 để làm gợi ý.")
        return pd.DataFrame()

    agg_scores = {}
    contribution_count = {}

    for movie_id in liked_movies['movieId'].values:
        movie_title = movies[movies['movieId'] == movie_id]['title'].values
        if not movie_title:
            continue
        movie_title = movie_title[0]
        
        if movie_title not in indices:
            continue
            
        idx = indices[movie_title]
        sim_scores = cosine_sim[idx].flatten()
        sim_scores = list(enumerate(sim_scores))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
        
        for i, score in sim_scores:
            if i in agg_scores:
                agg_scores[i] += score
                contribution_count[i] += 1
            else:
                agg_scores[i] = score
                contribution_count[i] = 1

    normalized_scores = []
    for i in agg_scores:
        avg_score = agg_scores[i] / contribution_count[i]
        normalized_scores.append((i, avg_score))
    
    sorted_scores = sorted(normalized_scores, key=lambda x: x[1], reverse=True)[:top_n]
    movie_indices = [i[0] for i in sorted_scores]
    scores = [i[1] for i in sorted_scores]
    
    result = movies.iloc[movie_indices][['movieId', 'title']].copy()
    result['similarity'] = scores
    
    return result


In [ ]:
user_id = random.choice(ratings['userId'].unique())
liked_movies = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4)]

if not liked_movies.empty:
    print(f"\nUser {user_id} thích các phim sau:")
    for movie_id in liked_movies['movieId'].values:
        movie_title = movies[movies['movieId'] == movie_id]['title'].values[0]
        print(f"- {movie_title}")
    
    cbf_recs = recommend_movies_for_user(user_id, ratings, movies, indices, cosine_sim, top_n=10)
    
    if not cbf_recs.empty:
        print("\nTop phim gợi ý (CBF):")
        print(cbf_recs)
        
        # Visualize recommendations
        fig = px.bar(
            cbf_recs.sort_values("similarity"),
            x="similarity",
            y="title",
            orientation="h",
            title=f"Top 10 phim gợi ý cho user {user_id} dựa trên các phim đã thích",
            labels={"similarity": "Độ tương đồng (0-1)", "title": "Tên phim"},
            color="similarity",
            color_continuous_scale="viridis"
        )
        fig.update_layout(yaxis={'categoryorder': 'total ascending'})
        fig.show()
    else:
        print("Không thể tạo gợi ý do không tìm thấy phim phù hợp.")
else:
    print(f"User {user_id} không có đánh giá >= 4 để làm gợi ý.")


User 411 thích: Toy Story (1995)

Top phim gợi ý (CBF):
      movieId                                           title  similarity
1757     2355                            Bug's Life, A (1998)    0.862225
2355     3114                              Toy Story 2 (1999)    0.644038
8695   122918                Guardians of the Galaxy 2 (2017)    0.367650
1706     2294                                     Antz (1998)    0.357912
2809     3754  Adventures of Rocky and Bullwinkle, The (2000)    0.357912
3000     4016                Emperor's New Groove, The (2000)    0.357912
3568     4886                           Monsters, Inc. (2001)    0.357912
6194    45074                                Wild, The (2006)    0.357912
6486    53121                          Shrek the Third (2007)    0.357912
6948    65577                  Tale of Despereaux, The (2008)    0.357912


In [10]:
def get_cf_score(user_id, movie_id, model):
    try:
        return model.predict(uid=str(user_id), iid=str(movie_id)).est
    except:
        return 0

def get_cbf_score(title_input, movie_id, cosine_sim, indices, movies_df):
    if title_input not in indices:
        return 0

    try:
        idx = indices[title_input]
        target_idx = movies_df[movies_df['movieId'] == movie_id].index[0]
        return cosine_sim[idx, target_idx]
    except:
        return 0


def normalize_scores(scores):
    scores = np.array(scores)
    if len(scores) == 0 or scores.max() == scores.min():
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

In [ ]:
def hybrid_recommend(user_id, title, model_cf, cosine_sim, indices, movies_df, ratings_df, alpha=0.7, top_n=10, candidate_limit=None):
    rated_movies = set(ratings_df[ratings_df['userId'] == user_id]['movieId'])

    candidate_movies = movies_df.copy()
    
    if candidate_limit:
        candidate_movies = candidate_movies.sample(n=candidate_limit, random_state=42)

    if title not in indices:
        print(f"Không tìm thấy phim '{title}' trong danh sách.")
        return pd.DataFrame()  
        if movie_id == original_movie_id:
            continue

        cf_score = get_cf_score(user_id, movie_id, model_cf)
        cbf_score = get_cbf_score(title, movie_id, cosine_sim, indices, movies_df)

        hybrid_scores.append((movie_id, movie_title, cf_score, cbf_score))

    cf_scores = normalize_scores([s[2] for s in hybrid_scores])
    cbf_scores = normalize_scores([s[3] for s in hybrid_scores])

    combined = []
    for (movie_id, title, _, _), cf_s, cbf_s in zip(hybrid_scores, cf_scores, cbf_scores):
        score = alpha * cf_s + (1 - alpha) * cbf_s
        combined.append((movie_id, title, score))

    combined.sort(key=lambda x: x[2], reverse=True)

    return pd.DataFrame(combined[:top_n], columns=['movieId', 'title', 'hybrid_score'])

In [12]:
def recommends_hybrid(user_id, recommended_df, ratings_df):
    rated = ratings_df[(ratings_df['userId'] == user_id) & (ratings_df['movieId'].isin(recommended_df['movieId']))]
    merged = recommended_df.merge(rated[['movieId', 'rating']], on='movieId', how='left').dropna()

    if merged.empty:
        return None

    y_true = merged['rating']
    y_pred = merged['hybrid_score'] * 5.0

    def mape(y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        mask = y_true != 0
        return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape_val = mape(y_true, y_pred)

    if len(y_true) >= 2:
        r2 = r2_score(y_true, y_pred)
    else:
        r2 = None  # hoặc 'N/A'

    relevant = set(rated[rated['rating'] >= 4]['movieId'])
    recommended = set(recommended_df['movieId'])
    hits = len(relevant.intersection(recommended))
    precision_at_k = hits / len(recommended) if recommended else 0

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape_val,
        "R²": r2,
        "Precision@K": precision_at_k
    }


In [13]:
user_id = random.choice(ratings['userId'].unique())

liked = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4)]
title = movies_tagged[movies_tagged['movieId'] == liked.iloc[0]['movieId']]['title'].values[0]

print("Gợi ý phim cho user:", user_id)
print(f"Phim đã thích: {liked['movieId'].values[0]} - {title}")

hybrid_df = hybrid_recommend(
    user_id=user_id,
    title=title,
    model_cf=model_cf,
    cosine_sim=cosine_sim,
    indices=indices,
    movies_df=movies_tagged,
    ratings_df=ratings,
    alpha=0.7,
    top_n=10,
    candidate_limit=None
)

eval_results = recommends_hybrid(user_id, hybrid_df, ratings)

print("Hybrid Results:")
print(hybrid_df)

print("\nEvaluation Metrics:")
print(eval_results)


Gợi ý phim cho user: 397
Phim đã thích: 123 - Chungking Express (Chung Hing sam lam) (1994)


Tính điểm Hybrid: 100%|██████████| 9742/9742 [00:09<00:00, 1056.97it/s]

Hybrid Results:
   movieId                                    title  hybrid_score
0      451                    Flesh and Bone (1993)      0.300000
1     2875                         Sommersby (1993)      0.300000
2     4146         Million Dollar Hotel, The (2001)      0.300000
3     6789  Apartment, The (Appartement, L') (1996)      0.300000
4     8195    Avventura, L' (Adventure, The) (1960)      0.300000
5   100527                        Safe Haven (2013)      0.300000
6   128512                       Paper Towns (2015)      0.300000
7   170355                    Mulholland Dr. (1999)      0.300000
8     1484                  Daytrippers, The (1996)      0.280044
9      427                     Boxing Helena (1993)      0.266258

Evaluation Metrics:
None


In [14]:
fig = px.bar(
    hybrid_df.sort_values("hybrid_score"),
    x="hybrid_score",
    y="title",
    orientation="h",
    title=f"Top {len(hybrid_df)} phim gợi ý (Kết hợp CF + CBF)",
    labels={"hybrid_score": "Điểm Hybrid", "title": "Tên phim"},
    color="hybrid_score",
    color_continuous_scale="viridis"
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title="Điểm kết hợp (0 → 1)",
    yaxis_title="Tên phim",
    font=dict(size=14)
)

fig.show()

In [15]:
def evaluate_hybrid_all_users(
    model_cf,
    ratings_df,
    movies_df,
    cosine_sim,
    indices,
    alpha=0.7,
    top_n=10,
    candidate_limit=None,
    n_users=30
):
    user_ids = ratings_df['userId'].unique()
    sampled_users = random.sample(list(user_ids), min(n_users, len(user_ids)))

    metrics = {
        'RMSE': [],
        'MAE': [],
        'MAPE': [],
        'R²': [],
        'Precision@K': []
    }

    for user_id in sampled_users:
        liked = ratings_df[(ratings_df['userId'] == user_id) & (ratings_df['rating'] >= 4)]
        if liked.empty:
            continue

        try:
            title = movies_df[movies_df['movieId'] == liked.iloc[0]['movieId']]['title'].values[0]
        except IndexError:
            continue

        hybrid_df = hybrid_recommend(
            user_id=user_id,
            title=title,
            model_cf=model_cf,
            cosine_sim=cosine_sim,
            indices=indices,
            movies_df=movies_df,
            ratings_df=ratings_df,
            alpha=alpha,
            top_n=top_n,
            candidate_limit=candidate_limit
        )

        eval_result = recommends_hybrid(user_id, hybrid_df, ratings_df)

        if eval_result:
            for key in metrics:
                value = eval_result[key]
                if value is not None:
                    metrics[key].append(value)

    # Tính trung bình
    avg_result = {key: (np.mean(metrics[key]) if metrics[key] else None) for key in metrics}

    print(f"\nTrung bình đánh giá trên {len(metrics['RMSE'])} user:")
    for key, value in avg_result.items():
        print(f"{key}: {value:.4f}" if value is not None else f"{key}: N/A")

    return avg_result


In [16]:
evaluate_hybrid_all_users(
    model_cf=model_cf,
    ratings_df=ratings,
    movies_df=movies_tagged,
    cosine_sim=cosine_sim,
    indices=indices,
    alpha=0.7, 
    top_n=10,
    candidate_limit=None,
    n_users=5
)


Tính điểm Hybrid: 100%|██████████| 9742/9742 [00:07<00:00, 1276.09it/s]


Trung bình đánh giá trên 3 user:
RMSE: 2.5942
MAE: 2.5912
MAPE: 63.2398
R²: -29.9747
Precision@K: 0.0667


{'RMSE': 2.5942488594240243,
 'MAE': 2.591171510172926,
 'MAPE': 63.239820925575664,
 'R²': -29.974714075539715,
 'Precision@K': 0.06666666666666667}